# Fake Review Detection

This notebook is a cleaned-up and reproducible version of the original project. It keeps the strongest modeling approach, removes unnecessary heavy dependencies, and presents the workflow in a form that is easier to explain in an ML project review. It also includes an optional `Qwen/Qwen2.5-1.5B-Instruct` comparison for custom reviews.

## Why this version is stronger

- Uses a single, high-performing text pipeline instead of mixing in weaker components
- Adds a proper train/validation/test split
- Saves metrics and charts for reporting
- Produces a reusable model for future predictions
- Lets you compare the ML prediction with `Qwen/Qwen2.5-1.5B-Instruct`

In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

ROOT = Path.cwd()
DATA_PATH = ROOT / 'fake_reviews_dataset.csv'
MODEL_DIR = ROOT / 'models'
OUTPUT_DIR = ROOT / 'outputs'
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]
df = df.dropna(subset=['text_', 'label']).copy()
df['text_'] = df['text_'].astype(str).str.strip()
df = df[df['text_'] != ''].copy()
df['target'] = df['label'].astype(str).str.strip().map({'CG': 1, 'OR': 0})
df = df.dropna(subset=['target']).copy()
df['target'] = df['target'].astype(int)
df.head()

In [ ]:
summary = pd.DataFrame({
    'rows': [len(df)],
    'unique_categories': [df['category'].nunique()],
    'avg_review_length': [round(df['text_'].str.len().mean(), 2)],
    'fake_count': [int(df['target'].sum())],
    'genuine_count': [int((1 - df['target']).sum())],
})
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#5B8FF9', '#61DDAA'])
axes[0].set_title('Label Distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')

df['text_'].str.len().plot(kind='hist', bins=40, ax=axes[1], color='#F6BD16')
axes[1].set_title('Review Length Distribution')
axes[1].set_xlabel('Characters')
plt.tight_layout()

In [ ]:
X = df['text_']
y = df['target']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
)

print(f'Train size: {len(X_train):,}')
print(f'Validation size: {len(X_val):,}')
print(f'Test size: {len(X_test):,}')

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=8000,
        min_df=3,
        sublinear_tf=True,
    )),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)
val_pred = model.predict(X_val)
val_score = model.predict_proba(X_val)[:, 1]
test_pred = model.predict(X_test)
test_score = model.predict_proba(X_test)[:, 1]

metrics = {
    'validation_accuracy': round(accuracy_score(y_val, val_pred), 4),
    'validation_f1': round(f1_score(y_val, val_pred), 4),
    'validation_auc': round(roc_auc_score(y_val, val_score), 4),
    'test_accuracy': round(accuracy_score(y_test, test_pred), 4),
    'test_f1': round(f1_score(y_test, test_pred), 4),
    'test_auc': round(roc_auc_score(y_test, test_score), 4),
}
metrics

In [ ]:
print(classification_report(
    y_test,
    test_pred,
    target_names=['Original / Genuine', 'Computer-Generated / Fake'],
    digits=4,
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred,
    display_labels=['Original / Genuine', 'Computer-Generated / Fake'],
    cmap='Blues',
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title('Confusion Matrix')

fpr, tpr, _ = roc_curve(y_test, test_score)
axes[1].plot(fpr, tpr, linewidth=2)
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title(f'ROC Curve (AUC = {roc_auc_score(y_test, test_score):.4f})')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')

plt.tight_layout()

In [ ]:
joblib.dump(model, MODEL_DIR / 'fake_review_detector.joblib')
with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('Saved model and metrics.')

In [ ]:
examples = [
    'BEST PRODUCT EVER!!! AMAZING QUALITY!!! BUY NOW!!!',
    'The item arrived in two days and matches the photos. The zipper feels solid.',
    'Good product. Recommended.',
]

for text in examples:
    probability = model.predict_proba([text])[0, 1]
    if probability >= 0.6:
        label = 'Likely computer-generated / fake'
    elif probability <= 0.4:
        label = 'Likely original / genuine'
    else:
        label = 'Borderline / uncertain'
    print(f'\nText: {text}')
    print(f'Fake probability: {probability:.4f}')
    print(f'Prediction: {label}')

## Optional LLM comparison

For a custom review, you can also compare the saved ML model with `Qwen/Qwen2.5-1.5B-Instruct`. The command below prints both results side by side.

```powershell
python predict_review.py --use-llm --text "BEST PRODUCT EVER!!! AMAZING QUALITY!!! BUY NOW!!!"
```